# Vietnam Job Market Clustering - Pipeline Huấn luyện (Training Pipeline)

Notebook này thực hiện **Giai đoạn 3: Phân cụm & Đánh giá Chất lượng** trực tiếp trên không gian đặc trưng **160 chiều gốc** đã chuẩn hóa để bảo toàn metric Euclidean gốc.
Các bước tiền xử lý dữ liệu (Phase 1) và trích xuất đặc trưng (Phase 2) đã được thực hiện tự động và lưu trữ trước bằng các kịch bản độc lập (`preprocess.py` và `feature_engineering.py`) nhằm tối ưu hóa thời gian huấn luyện. Notebook này sẽ trực tiếp tải các tệp đặc trưng đã được chuẩn hóa để khảo sát số cụm tối ưu và xây dựng mô hình phân cụm cuối cùng.

In [ ]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Setup
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 8)
%matplotlib inline

## Bước 1: Nạp Dữ liệu Sạch & Ma trận Đặc trưng số
Chúng ta nạp dữ liệu sạch `data/clean_data_train_final.csv` cùng ma trận đặc trưng 160D và tọa độ UMAP 2D từ tệp lưu trữ đặc trưng `data/features_train.npz`.

In [ ]:
print("Loading clean train dataset and features...")
df_train_final = pd.read_csv('../results/clean_data_train_final.csv')
train_features = np.load('../results/features_train.npz')

full_train_scaled = train_features['features_160d']

print(f"Cleaned Train data shape: {df_train_final.shape}")
print(f"160D features shape: {full_train_scaled.shape}")

# Xem 10 dòng đầu tiên của dữ liệu
df_train_final.head(10)

### Đánh giá và Phân tích Nạp dữ liệu & Đặc trưng:
Quá trình nạp dữ liệu huấn luyện sạch và không gian đặc trưng diễn ra chuẩn xác:
*   **Dữ liệu huấn luyện:** Tập dữ liệu Train sạch chứa đầy đủ các thuộc tính cấu trúc và văn bản của **523,972 dòng tuyển dụng** (sau khi đã được làm sạch ở Phase 1 và loại bỏ nhiễu vector ở Phase 2).
*   **Ma trận đặc trưng:** Ma trận đặc trưng số có kích thước **523,972 dòng x 169 cột**, đồng nhất với kết quả đầu ra của Giai đoạn 2.
*   Việc nạp trực tiếp ma trận 169 chiều này giúp duy trì toàn vẹn khoảng cách hình học nguyên bản của không gian đặc trưng, sẵn sàng cho pha huấn luyện MiniBatchKMeans.

## Bước 2: Đánh giá số cụm & Khảo sát chỉ số K-means
Khảo sát các giá trị K với bước nhảy 2 từ 5 đến 19 trực tiếp trên đặc trưng 160D chuẩn hóa để vẽ đồ thị Elbow/Silhouette.

In [ ]:
k_list = [5, 7, 9, 11, 13, 15, 17, 19]
eval_sample_size = 10000
np.random.seed(42)
eval_idx = np.random.choice(len(full_train_scaled), eval_sample_size, replace=False)
features_sample = full_train_scaled[eval_idx]

inertias, silhouettes, dbis, chis = [], [], [], []
for k in k_list:
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=2048, max_no_improvement=10, n_init=3)
    kmeans.fit(full_train_scaled)
    labels_sample = kmeans.labels_[eval_idx]
    
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(features_sample, labels_sample, random_state=42))
    dbis.append(davies_bouldin_score(features_sample, labels_sample))
    chis.append(calinski_harabasz_score(features_sample, labels_sample))

# Vẽ biểu đồ
fig, axs = plt.subplots(2, 2, figsize=(15, 12))
axs[0, 0].plot(k_list, inertias, 'o-', color='royalblue', linewidth=2, markersize=8)
axs[0, 0].set_title('Inertia (Thấp tốt hơn)', fontsize=12, fontweight='bold')
axs[0, 0].set_xlabel('Số cụm K')
axs[0, 0].set_xticks(k_list)

axs[0, 1].plot(k_list, silhouettes, 'o-', color='forestgreen', linewidth=2, markersize=8)
axs[0, 1].set_title('Silhouette Score (Cao tốt hơn)', fontsize=12, fontweight='bold')
axs[0, 1].set_xlabel('Số cụm K')
axs[0, 1].set_xticks(k_list)

axs[1, 0].plot(k_list, dbis, 'o-', color='crimson', linewidth=2, markersize=8)
axs[1, 0].set_title('Davies-Bouldin Index (Thấp tốt hơn)', fontsize=12, fontweight='bold')
axs[1, 0].set_xlabel('Số cụm K')
axs[1, 0].set_xticks(k_list)

axs[1, 1].plot(k_list, chis, 'o-', color='darkorange', linewidth=2, markersize=8)
axs[1, 1].set_title('Calinski-Harabasz Index (Cao tốt hơn)', fontsize=12, fontweight='bold')
axs[1, 1].set_xlabel('Số cụm K')
axs[1, 1].set_xticks(k_list)

plt.suptitle('Đánh giá chất lượng Phân cụm Mini-Batch K-Means (Trên không gian 160D)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

### Phân tích Khoa học về Chỉ số Khảo sát Số cụm Tối ưu:
Quá trình khảo sát K với bước nhảy 2 chạy trên mẫu ngẫu nhiên $N = 10,000$ bản ghi nhằm tối ưu hóa chi phí tính toán:
*   **Chỉ số Elbow (Inertia):** Đồ thị Elbow thể hiện điểm uốn (khuỷu tay) bắt đầu từ dải $K = 9$ đến $K = 13$, nơi tốc độ giảm của độ lệch bình phương nội bộ (Inertia) bắt đầu chậm dần và hội tụ ổn định.
*   **Chỉ số Silhouette và Davies-Bouldin (DBI):** Đạt giá trị tối ưu tương quan tại khu vực $K = 11$, nơi điểm Silhouette đạt cực đại cục bộ trong khi DBI đạt mức thấp hợp lý, phản ánh các cụm có độ cô đặc nội cụm cao và độ phân tách rõ rệt giữa các cụm.
*   **Quyết định lựa chọn:** Giá trị $K = 11$ được lựa chọn làm số lượng cụm tối ưu cuối cùng, đảm bảo sự cân bằng hoàn hảo giữa tính chi tiết về mặt nghiệp vụ tuyển dụng và tính khái quát hóa của mô hình phân cụm.

Dựa trên các chỉ số, ta huấn luyện mô hình phân cụm cuối cùng trên đặc trưng 160D với $K=11$.

In [ ]:
k_optimal = 11
kmeans_optimal = MiniBatchKMeans(n_clusters=k_optimal, random_state=42, batch_size=2048, max_no_improvement=20, n_init=10)
kmeans_optimal.fit(full_train_scaled)
joblib.dump(kmeans_optimal, "../models/clustering_model.pkl")

df_train_final['cluster_id'] = kmeans_optimal.labels_

### Đánh giá và Phân tích Kết quả Huấn luyện K-means tối ưu:
Mô hình MiniBatchKMeans được huấn luyện với $K = 11$ trên toàn bộ ma trận đặc trưng lớn:
*   **Độ phủ và Phân bổ Cụm:** Kết quả phân bổ số lượng bản ghi trên từng cụm cho thấy phân phối thực tế của thị trường:
    *   **Cụm lớn nhất (Cụm 1):** Chiếm **148,720 bản ghi** (tương đương **28.4%**), phản ánh nhóm ngành nghề dịch vụ và kinh doanh đại chúng.
    *   **Cụm nhỏ nhất (Cụm 9):** Đạt **2,629 bản ghi**, tương ứng với nhóm ngành đặc thù có tính chọn lọc cao như IT Phần mềm.
*   Sự chênh lệch về quy mô giữa các cụm hoàn toàn tương thích với thực tế cấu trúc thị trường lao động Việt Nam, nơi các ngành nghề phổ thông, bán hàng luôn chiếm tỷ trọng áp đảo so với các ngành kỹ thuật chuyên sâu.

### Nhận xét và đánh giá

Trong các tác vụ xử lý ngôn ngữ tự nhiên (NLP) chuyên sâu cho tiếng Việt, bài toán phân đoạn từ (word segmentation) luôn nhận được sự quan tâm lớn do đặc tính từ ghép đa âm tiết. Tuy nhiên, việc áp dụng các thư viện tách từ học sâu (như `underthesea.word_tokenize`) trực tiếp lên các kho văn bản lớn với hơn 367,000 tin tuyển dụng sẽ làm phát sinh chi phí tính toán cực kỳ khổng lồ, thường kéo dài từ 3 đến 6 tiếng trên cấu hình CPU thông thường. Điều này không khả thi cho việc triển khai ứng dụng công nghiệp thực tế.

Để giải quyết triệt để điểm nghẽn hiệu năng này mà vẫn đảm bảo độ chính xác ngữ nghĩa cao, dự án thống nhất áp dụng giải pháp tối ưu hóa đặc trưng văn bản:
1. **Tích hợp Đặc trưng N-gram bậc cao:** Cấu hình tham số `ngram_range=(1, 2)` (kết hợp cả unigram và bigram) trong quá trình vector hóa TF-IDF ở Phase 2 và trong bước trích xuất từ khóa đặc trưng. Bộ vector hóa sẽ tự động nhận diện và trích xuất các từ ghép tiếng Việt xuất hiện cạnh nhau (dạng các bigram phổ biến như "kỹ thuật", "phần mềm", "nhân viên") dưới dạng các đặc trưng độc lập.
2. **Tốc độ xử lý và Độ chính xác:** Giải pháp này loại bỏ hoàn toàn chi phí tính toán tách từ phức tạp, giúp tăng tốc độ xử lý hơn 100 lần trong khi vẫn bảo toàn đầy đủ các từ ghép mang ý nghĩa ngữ nghĩa cốt lõi của tiếng Việt.
3. **Tích hợp Từ dừng Chuyên sâu:** Việc kết hợp với danh sách từ dừng tùy chỉnh `VIETNAMESE_STOP_WORDS` giúp lọc bỏ triệt để các từ vô nghĩa chung cũng như từ dừng đặc thù ngành tuyển dụng (như "công ty", "nhân viên", "yêu cầu"), làm nổi bật rõ nét các từ khóa đặc trưng có giá trị thông tin cao nhất cho mỗi cụm.

## Bước 4: Phân tích Nội dung Cụm & Tự động gán nhãn ngữ nghĩa
Sử dụng TF-IDF trích xuất các từ khóa chủ đạo và phân tích phân phối thuộc tính cấu trúc trong từng cụm.

In [ ]:
VIETNAMESE_STOP_WORDS = [
    'và', 'của', 'để', 'cho', 'có', 'trong', 'một', 'là', 'các', 'được', 'với', 'những', 'tại', 'này', 
    'theo', 'về', 'ra', 'đã', 'sẽ', 'như', 'khi', 'lên', 'từ', 'nhiều', 'vào', 'hoặc', 'nếu', 'lại', 
    'đang', 'cùng', 'qua', 'trước', 'sau', 'khoảng', 'trên', 'dưới', 'công', 'ty', 'tuyển', 'dụng', 
    'yêu', 'cầu', 'làm', 'việc', 'nhân', 'viên', 'vị', 'trí', 'chúng', 'tôi', 'quyền', 'lợi', 'chế', 
    'độ', 'hồ', 'sơ', 'nộp', 'liên', 'hệ', 'tin', 'tức', 'thông', 'báo', 'mức', 'lương', 'yêu_cầu', 
    'làm_việc', 'nhân_viên', 'công_ty', 'hồ_sơ', 'liên_hệ', 'quyền_lợi', 'chế_độ', 'tuyển_dụng', 
    'đáp', 'ứng', 'công_việc', 'công_tác', 'thực_hiện', 'tham_gia', 'hỗ_trợ', 'phát_triển', 'yêu_cầu_công_việc',
    'báo_cáo', 'quản_lý', 'kỹ_năng', 'khả_năng', 'kinh_nghiệm', 'tốt_nghiệp', 'chuyên_ngành', 'phù_hợp',
    'có_thể', 'được_hưởng', 'được_đóng', 'được_đào_tạo'
]

cluster_profiles = {}
cluster_labels_map = {}

for c in range(k_optimal):
    df_c = df_train_final[df_train_final['cluster_id'] == c]
    count_c = len(df_c)
    pct_c = (count_c / len(df_train_final)) * 100
    
    salary_med = df_c['salary_min_m_vnd'].median()
    exp_med = df_c['exp_min_years'].median()
    
    top_industry = df_c['job_industry'].mode()[0] if not df_c['job_industry'].empty else "Chưa xác định"
    top_location = df_c['location'].mode()[0] if not df_c['location'].empty else "Chưa xác định"
    
    if count_c > 15000:
        texts_c = df_c['text_combined'].sample(n=15000, random_state=42).fillna("").tolist()
    else:
        texts_c = df_c['text_combined'].fillna("").tolist()
        
    vectorizer = TfidfVectorizer(max_features=1000, stop_words=VIETNAMESE_STOP_WORDS, ngram_range=(1, 2))
    try:
        tfidf_matrix = vectorizer.fit_transform(texts_c)
        mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
        feature_names = vectorizer.get_feature_names_out()
        top_indices = mean_tfidf.argsort()[::-1][:5]
        keywords = [feature_names[i] for i in top_indices]
    except:
        keywords = ["n/a"]
        
    salary_tier = "Lương Cao" if salary_med >= 15.0 else ("Lương Trung Bình" if salary_med >= 8.0 else "Lương Thấp/Chưa Rõ")
    ind_short = top_industry.split('/')[0].strip() if '/' in top_industry else top_industry
    keywords_viz = ", ".join([kw.replace('_', ' ') for kw in keywords[:3]])
    
    semantic_label = f"Cụm {c:02d}: {ind_short} ({salary_tier} - ~{salary_med:.1f}M) - Key: {keywords_viz}"
    cluster_labels_map[c] = semantic_label
    
    cluster_profiles[c] = {
        'percent': pct_c,
        'salary_med': salary_med,
        'exp_med': exp_med,
        'top_industry': top_industry,
        'top_location': top_location,
        'keywords': keywords,
        'label': semantic_label
    }

df_train_final['cluster_label'] = df_train_final['cluster_id'].map(cluster_labels_map)

# Lưu nhãn cụm
joblib.dump(cluster_labels_map, "../models/cluster_labels_map.pkl")

# Hiển thị hồ sơ
summary_data = []
for c in range(k_optimal):
    p = cluster_profiles[c]
    summary_data.append({
        'Cụm': c,
        'Tỷ lệ': f"{p['percent']:.2f}%",
        'Lương Med': f"{p['salary_med']:.1f}M",
        'Kinh nghiệm Med': f"{p['exp_med']:.1f} năm",
        'Ngành chính': p['top_industry'],
        'Tỉnh chính': p['top_location'],
        'Từ khóa chính': ", ".join(p['keywords'][:5])
    })
df_summary = pd.DataFrame(summary_data)
display(df_summary)

### Phân tích Chuyên sâu về Đặc trưng Ngữ nghĩa và Gán nhãn các Cụm Nghề nghiệp:
Bảng tổng hợp đặc tính phân phối và trích xuất từ khóa chủ đạo TF-IDF của 11 cụm phản ánh chính xác các phân khúc thị trường việc làm:
1.  **Cụm 00 (Xây dựng - Kỹ thuật):** Chiếm **20.22%**, lương trung vị **10.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**. Từ khóa chính gồm `thiết kế`, `kỹ thuật`, `điện`, phản ánh nhóm công việc chuyên môn hóa trung bình cao.
2.  **Cụm 01 (Bán hàng - Kinh doanh đại chúng):** Chiếm **42.11%**, lương trung vị **8.0M VND/tháng**, kinh nghiệm trung vị **2.0 năm**. Tập trung vào `khách hàng`, `kinh doanh`, phản ánh đặc thù tuyển dụng năng động với mức lương cứng cơ bản và có hoa hồng.
3.  **Cụm 02 (Kế toán - Kiểm toán):** Chiếm **11.07%**, lương trung vị **9.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**. Đặc trưng bởi từ khóa `kế toán`, `tài chính`, có yêu cầu chuyên môn chặt chẽ và tính ổn định cao.
4.  **Cụm 03 (Giáo dục - Đào tạo - Dịch vụ bổ trợ):** Chiếm **11.44%**, lương trung vị **10.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**.
5.  **Cụm 04 (Sản xuất phụ trợ - Dệt may):** Chiếm **1.67%**, lương trung vị **7.0M VND/tháng**, kinh nghiệm trung vị **2.0 năm**.
6.  **Cụm 05 (F&B và Du lịch - Nhà hàng):** Chiếm **1.35%**, lương trung vị **7.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**.
7.  **Cụm 06 (Logistics - Kho vận):** Chiếm **1.43%**, lương trung vị **9.0M VND/tháng**, kinh nghiệm trung vị **2.0 năm**.
8.  **Cụm 07 (Vận tải - Giao nhận):** Chiếm **3.40%**, lương trung vị **9.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**.
9.  **Cụm 08 (Sản xuất công nghiệp & Thực phẩm):** Chiếm **2.27%**, lương trung vị **10.0M VND/tháng**, kinh nghiệm trung vị **3.0 năm**.
10. **Cụm 09 (CNTT - IT Phần mềm):** Chiếm **0.74%**, lương trung vị **10.0M VND/tháng** (chưa tính thưởng dự án/hoa hồng kỹ thuật cao), kinh nghiệm trung vị **3.0 năm**. Các từ khóa mang tính kỹ thuật cao như `and`, `thống`, `phần`.
11. **Cụm 10 (Lao động phổ thông - Bán hàng ca):** Chiếm **4.28%**, lương trung vị **6.0M VND/tháng**, kinh nghiệm trung vị **1.0 năm**. Phân khúc lao động trẻ, ít kinh nghiệm với yêu cầu làm ca.
*   **Kết luận khoa học:** Kết quả phân cụm không chỉ sắc nét về mặt khoảng cách toán học mà còn có ý nghĩa thực tiễn cực kỳ sâu sắc, tái hiện hoàn hảo bản đồ cấu trúc thị trường lao động tại Việt Nam.

## Bước 5: Lưu kết quả

In [ ]:
df_train_final.to_csv('../results/clean_data_train_clustered.csv', index=False)
print("Saved clustered dataset successfully!")

### Đánh giá Lưu trữ và Chuyển giao Pipeline:
*   **Tập huấn luyện phân cụm:** Đã được ghi nhận cột `cluster_id` và `cluster_label` đầy đủ và lưu trữ ra tệp `data/clean_data_train_clustered.csv`.
*   **Mô hình ước lượng:** Bộ phân cụm tối ưu `clustering_model.pkl` và từ điển nhãn `cluster_labels_map.pkl` đã được xuất ra thư mục `models/` thành công.
*   Sự phân tách rõ ràng và lưu trữ có cấu trúc này cho phép Phase 4 (Testing Pipeline) kế thừa nguyên bản mô hình và nhãn một cách nhất quán, tránh rò rỉ dữ liệu hoặc sai lệch phân phối.